### Aula 05: Análise de Regras de Associação para Produtos de Supermercado

Este notebook demonstra como usar o algoritmo **Apriori** para descobrir padrões de compra em transações de supermercado. O objetivo é identificar combinações de produtos que tendem a ser compradas juntas, o que pode ser útil para estratégias de marketing e organização de prateleiras.

#### 1\. Instalação e Importação de Bibliotecas

Primeiro, você precisa instalar a biblioteca `mlxtend` que contém o algoritmo Apriori. As bibliotecas `pandas` e `io` também são necessárias.

```bash
pip install pandas mlxtend
```

In [1]:
!pip install pandas mlxtend

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 23.2 MB/s eta 0:00:00


In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
from mlxtend.preprocessing import TransactionEncoder

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


#### 2\. Carregamento e Pré-processamento dos Dados

Para que o algoritmo Apriori funcione, a lista de transações deve ser convertida em um formato de "tabela de transações" (também conhecido como **one-hot encoding**), onde cada item é uma coluna e o valor `True` indica que o item está presente na transação.

In [2]:
# Carregando o DataFrame a partir do CSV
df = pd.read_csv('datasets/produtos.csv')

# Preparando os dados no formato de lista de listas
transacoes = []
for _, row in df.iterrows():
    items = row['produtos_comprados'].split(';')
    items = [item.strip() for item in items]
    transacoes.append(items)

print("Dados em formato de lista de transações:")
print(transacoes)

# Convertendo para o formato one-hot encoded
te = TransactionEncoder()
te_ary = te.fit(transacoes).transform(transacoes)
df_onehot = pd.DataFrame(te_ary, columns=te.columns_)

print("\nDataFrame one-hot encoded:")
print(df_onehot)

Dados em formato de lista de transações:
[['biscoito', 'cerveja', 'chá', 'salaminho'], ['cerveja', 'couve', 'linguiça', 'pão', 'queijo'], ['café', 'brócolis', 'couve', 'pão'], ['brócolis', 'café', 'cerveja', 'couve', 'pão', 'salaminho'], ['brócolis', 'café', 'couve', 'pão', 'refrigerante'], ['couve', 'linguiça']]

DataFrame one-hot encoded:
   biscoito  brócolis   café  cerveja    chá  couve  linguiça    pão  queijo  \
0      True     False  False     True   True  False     False  False   False   
1     False     False  False     True  False   True      True   True    True   
2     False      True   True    False  False   True     False   True   False   
3     False      True   True     True  False   True     False   True   False   
4     False      True   True    False  False   True     False   True   False   
5     False     False  False    False  False   True      True  False   False   

   refrigerante  salaminho  
0         False       True  
1         False      False  
2        

In [3]:
df_onehot.head()

,biscoito,brócolis,café,cerveja,chá,couve,linguiça,pão,queijo,refrigerante,salaminho
0,True,False,False,True,True,False,False,False,False,False,True
1,False,False,False,True,False,True,True,True,True,False,False
2,False,True,True,False,False,True,False,True,False,False,False
3,False,True,True,True,False,True,False,True,False,False,True
4,False,True,True,False,False,True,False,True,False,True,False


#### 3\. Encontrando os Itemsets Frequentes

O algoritmo **Apriori** encontra todos os conjuntos de itens (itemsets) que aparecem em um número mínimo de transações. O parâmetro `min_support` define esse limite (ex: 0.5 significa que o itemset deve aparecer em pelo menos 50% das transações).

In [4]:
# Encontrando itemsets frequentes com suporte mínimo de 50%
frequent_itemsets = apriori(df_onehot, min_support=0.5, use_colnames=True)

print("\nItemsets frequentes (suporte >= 0.5):")
print(frequent_itemsets)


Itemsets frequentes (suporte >= 0.5):
     support                      itemsets
0   0.500000                    (brócolis)
1   0.500000                        (café)
2   0.500000                     (cerveja)
3   0.833333                       (couve)
4   0.666667                         (pão)
5   0.500000              (brócolis, café)
6   0.500000             (couve, brócolis)
7   0.500000               (pão, brócolis)
8   0.500000                 (couve, café)
9   0.500000                   (pão, café)
10  0.666667                  (pão, couve)
11  0.500000       (couve, brócolis, café)
12  0.500000         (pão, brócolis, café)
13  0.500000        (pão, couve, brócolis)
14  0.500000            (pão, couve, café)
15  0.500000  (pão, couve, brócolis, café)


#### 4\. Gerando Regras de Associação

A partir dos itemsets frequentes, podemos gerar as regras de associação. As regras têm um antecedente (seção "antecedents") e um consequente (seção "consequents"). O parâmetro `metric` pode ser `confidence`, `lift` ou `support`.

In [5]:
# Gerando as regras de associação com confiança mínima de 70%
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.7)

print("\nRegras de associação geradas (confiança >= 0.7):")
print(rules)


Regras de associação geradas (confiança >= 0.7):
                antecedents              consequents  antecedent support  \
0                (brócolis)                   (café)            0.500000   
1                    (café)               (brócolis)            0.500000   
2                (brócolis)                  (couve)            0.500000   
3                     (pão)               (brócolis)            0.666667   
4                (brócolis)                    (pão)            0.500000   
5                    (café)                  (couve)            0.500000   
6                     (pão)                   (café)            0.666667   
7                    (café)                    (pão)            0.500000   
8                     (pão)                  (couve)            0.666667   
9                   (couve)                    (pão)            0.833333   
10        (couve, brócolis)                   (café)            0.500000   
11            (couve, café)           

In [6]:
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']][(rules['confidence'] >= 0.8) & (rules['lift'] >= 1.5)]

,antecedents,consequents,support,confidence,lift
0,(brócolis),(café),0.5,1.0,2.0
1,(café),(brócolis),0.5,1.0,2.0
4,(brócolis),(pão),0.5,1.0,1.5
7,(café),(pão),0.5,1.0,1.5
10,"(couve, brócolis)",(café),0.5,1.0,2.0
11,"(couve, café)",(brócolis),0.5,1.0,2.0
13,(brócolis),"(couve, café)",0.5,1.0,2.0
14,(café),"(couve, brócolis)",0.5,1.0,2.0
15,"(pão, brócolis)",(café),0.5,1.0,2.0
16,"(pão, café)",(brócolis),0.5,1.0,2.0


#### 5\. Entendendo as Medidas (Suporte, Confiança e Lift)

As regras de associação são avaliadas por três métricas principais. Vamos entender o que cada uma significa com exemplos.

  * **Suporte (Support):**

      * O suporte de um itemset mede a frequência com que ele aparece no total de transações.
      * `support({couve})` = 4 transações (transações 2, 3, 4, 5) / 6 transações totais = **0.6667**
      * Isso significa que a `couve` está presente em 66,7% de todas as transações.

  * **Confiança (Confidence):**

      * A confiança de uma regra `A -> B` é a probabilidade de um cliente comprar `B` dado que ele já comprou `A`.
      * `confidence({brócolis} -> {café})` = Suporte(`{brócolis, café}`) / Suporte(`{brócolis}`)
      * No nosso caso, a regra `brócolis -> café` tem confiança de 1.000, o que significa que **sempre que o brócolis é comprado, o café também é**.

  * **Lift:**

      * O Lift de uma regra `A -> B` indica o quão mais provável é a compra de `B` quando `A` é comprado, comparado à probabilidade de `B` ser comprado sozinho.
      * `Lift({brócolis} -> {café})` = Confiança(`{brócolis} -> {café}`) / Suporte(`{café}`)
      * Se o **Lift \> 1**, há uma associação positiva (os itens são comprados juntos com mais frequência do que o esperado por acaso).
      * Se o **Lift = 1**, não há associação.
      * Se o **Lift \< 1**, há uma associação negativa.
      * No exemplo de `brócolis -> café`, o Lift é **1.5**, indicando uma associação forte e positiva. O `café` é 1,5 vezes mais provável de ser comprado junto com o `brócolis` do que por acaso.

**Conclusão:**
A análise de regras de associação é uma ferramenta poderosa para descobrir padrões ocultos em dados de transações. As métricas de suporte, confiança e lift fornecem insights valiosos sobre o comportamento dos consumidores, permitindo decisões de negócio mais inteligentes.